# UCS761 - Deep Learning Lab 4
### Multiple Linear Regression using a Linear Perceptron

Previous labs were about classification: predicting 0 or 1.
This one is different. The output is an actual number (salary), not a category.

Two things change because of this:
1. No activation function. We need the raw number, not a squashed probability.
2. Loss function changes from cross-entropy to MSE.

Dataset: age + experience to predict income

[Kaggle dataset link](https://www.kaggle.com/datasets/hussainnasirkhan/multiple-linear-regression-dataset)

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


## Step 1: Load and Explore

In [2]:
df = pd.read_csv("salary.csv")

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Sample data:")
print(df.head())


Shape: (20, 3)
Columns: ['age', 'experience', 'income']
Sample data:
   age  experience  income
0   25           1   30450
1   30           3   35670
2   47           2   31580
3   32           5   40130
4   43          10   47830


In [3]:
print("Stats:")
print(df.describe())


Stats:
             age  experience        income
count  20.000000   20.000000     20.000000
mean   39.650000    6.200000  40735.500000
std    10.027725    4.124382   8439.797625
min    23.000000    1.000000  27840.000000
25%    31.500000    3.750000  35452.500000
50%    40.000000    5.000000  40190.000000
75%    47.000000    9.000000  45390.000000
max    58.000000   17.000000  63600.000000


## Step 2: Split Features and Target

Input: age, experience
Output: income

In [4]:
X = df[["age", "experience"]].values
y = df["income"].values

print("X:", X.shape)
print("y:", y.shape)
print("First 3 X:", X[:3])
print("First 3 y:", y[:3])


X: (20, 2)
y: (20,)
First 3 X: [[25  1]
 [30  3]
 [47  2]]
First 3 y: [30450 35670 31580]


## Step 3: Train-Test Split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train size:", len(X_train))
print("Test size: ", len(X_test))


Train size: 16
Test size:  4


## Step 4: Scale the Data

income is in thousands or lakhs. age is single digits.
Without scaling, the gradients for these two features have totally different magnitudes and the model will not learn properly.

We scale y separately so we can inverse transform predictions back to real salary values.

In [6]:
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train = scaler_X.fit_transform(X_train)
X_test  = scaler_X.transform(X_test)

y_tr = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()
y_te = scaler_y.transform(y_test.reshape(-1, 1)).ravel()

print("Scaling done.")


Scaling done.


## Step 5: The Model: No Activation

y_hat = X @ w + b

No sigmoid, no step function. Just the raw linear combination. We want an actual number as output, not a probability or class label.

In [7]:
def predict(X, w, b):
    return X.dot(w) + b


## Step 6: MSE Loss

Mean Squared Error: penalizes based on how far the prediction is from the actual value.

L = (1/N) * sum((y_hat - y)^2)

We use MSE here and not cross-entropy because the output is a continuous number. Cross-entropy only makes sense when the output is a probability.

In [8]:
def mse(y_true, y_pred):
    return np.mean((y_pred - y_true) ** 2)


## Step 7: Gradients

Derivative of MSE with respect to weights:

dL/dw = (2/N) * X^T @ (y_hat - y)
dL/db = (2/N) * sum(y_hat - y)

These tell us which direction to move the weights to reduce loss.

In [9]:
def get_grads(X, y, y_hat):
    n  = len(y)
    dw = (2/n) * X.T.dot(y_hat - y)
    db = (2/n) * np.sum(y_hat - y)
    return dw, db


## Step 8: Parameter Update

In [10]:
def step_params(w, b, dw, db, lr):
    w = w - lr * dw
    b = b - lr * db
    return w, b


## Step 9: Training Loop

In [11]:
w    = np.zeros(X_train.shape[1])
b    = 0.0
lr   = 0.01
loss_log = []

for epoch in range(1000):
    y_hat    = predict(X_train, w, b)
    l        = mse(y_tr, y_hat)
    loss_log.append(l)
    dw, db   = get_grads(X_train, y_tr, y_hat)
    w, b     = step_params(w, b, dw, db, lr)

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1:4d}  Loss: {l:.6f}")

print(f"Final weights: {w}")
print(f"Final bias:    {b:.4f}")


Epoch  100  Loss: 0.068964
Epoch  200  Loss: 0.030694
Epoch  300  Loss: 0.026438
Epoch  400  Loss: 0.025944
Epoch  500  Loss: 0.025886
Epoch  600  Loss: 0.025879
Epoch  700  Loss: 0.025879
Epoch  800  Loss: 0.025879
Epoch  900  Loss: 0.025879
Epoch 1000  Loss: 0.025879
Final weights: [-0.10787613  1.03250895]
Final bias:    -0.0000


## Step 10: Evaluate

In [12]:
y_pred_scaled   = predict(X_test, w, b)
test_loss       = mse(y_te, y_pred_scaled)
print(f"Test MSE (scaled): {test_loss:.4f}")

y_pred_original = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()

print("Predicted vs Actual (first 5):")
for pred, actual in zip(y_pred_original[:5], y_test[:5]):
    print(f"  Predicted: {pred:>10.0f}   Actual: {actual:>10.0f}")


Test MSE (scaled): 0.0112
Predicted vs Actual (first 5):
  Predicted:      31093   Actual:      30450
  Predicted:      31295   Actual:      30870
  Predicted:      40250   Actual:      38900
  Predicted:      34898   Actual:      35670


In [13]:
new_person = np.array([[30, 5]])
new_scaled = scaler_X.transform(new_person)
pred_s     = predict(new_scaled, w, b)
pred_final = scaler_y.inverse_transform([[pred_s[0]]])[0][0]

print(f"New candidate: age=30, experience=5 years")
print(f"Predicted income: {pred_final:,.0f}")

New candidate: age=30, experience=5 years
Predicted income: 39,207


## Observations and Answers

**Why no activation function:**
Because the output is salary, an arbitrary number. Adding sigmoid would squash everything to (0,1). Adding step would give 0 or 1. Neither is useful when you want the actual salary value.

**Why MSE and not cross-entropy:**
Cross-entropy is for probability outputs (values between 0 and 1). income is not a probability. MSE measures how far the numeric prediction is from the actual value, which is what we want to minimize.

**How this is different from logistic regression:**
Logistic regression uses sigmoid, outputs probability, uses cross-entropy, gives a binary output.
Linear regression has no activation, outputs a raw number, uses MSE, gives a continuous output.
Core idea is the same (gradient descent updates weights) but the objective is completely different.

**Does loss decrease over time:**
It should. If it goes up, learning rate is too high. If it barely moves, try increasing lr a little.